<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Translate a need

**[Translate a need](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/)** · Machine Learning: From Problem to Reliable Model · Module 2, lesson 1 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn Larkfield's need into a prediction problem: the decision, the prediction unit, the target, the prediction time and the features. Then you look at the data in those terms.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Features and labels, from [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/#features-and-labels). The target and the prediction time, from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/) in the SQL course. |
| **You do not need** | A local project. The setup below downloads the data. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M02-L01-translate-a-need/translate-a-need-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Five questions before any model

A need is not yet a question that a model can answer. A baker who says "I throw away too much bread" needs a prediction of the loaves sold each day, made at five in the morning, from facts known at five. The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/#from-a-need-to-a-question) tells this story.

Grace, the support lead, says: "Some tickets need a senior agent. When we find out late, the customer has already waited a day. I want those tickets to go to the senior team when they arrive."

Priya, the data scientist, turns this need into five answers. Write them before you write any model code.

| Question | Larkfield's answer |
|---|---|
| **Decision:** what does a person do differently? | A flagged ticket goes straight to the senior team. The senior team can review at most 1 in 5 new tickets (20%). |
| **Prediction unit:** one prediction is for one … | ticket. |
| **Target:** what do we predict? | `escalated_72h`: 1 if a senior agent took the ticket within 72 hours of creation, else 0. |
| **Prediction time:** when must the answer exist? | When the ticket is created. |
| **Features:** what can the model use? | Only facts known at creation: the 10 columns from the SQL course. |

The [lesson page](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/#from-the-decision-to-the-metric) has a diagram of this chain. In this notebook, you look at the data through these five answers.

## 2. The data, in the decision's terms

Run the next cell. It defines `read()`, which reads a ticket file so that only an empty cell is missing (the text `unknown` stays a category). Then it loads the three parts of the split and prints their size, escalations and dates.

> **Predict.** About what share of tickets escalate: 1 in 2, 1 in 8 or 1 in 50? Then run the cell.

In [ ]:
def read(path):
    """Read a ticket file. Only an empty cell is missing: "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


train = read("data/train.csv")
valid = read("data/valid.csv")
test = read("data/test.csv")
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    first, last = part["created_at"].min().date(), part["created_at"].max().date()
    print(f"{name}: {len(part)} tickets, {part['escalated_72h'].sum()} escalated "
          f"({part['escalated_72h'].mean():.1%}), {first} to {last}")

> **Check.** You should see:
>
> ```
> train: 16939 tickets, 1983 escalated (11.7%), 2024-07-01 to 2026-04-30
> valid: 1179 tickets, 183 escalated (15.5%), 2026-05-01 to 2026-05-31
> test: 1002 tickets, 157 escalated (15.7%), 2026-06-01 to 2026-06-27
> ```

About 1 in 8 tickets escalate in train. In May and June, the rate is higher, because of the new warranty policy of 1 May. The split is the same rule as in the SQL course: train is the past, valid is May, test is June.

Grace's capacity is "1 in 5 new tickets". Run the next cell. It turns that into tickets per day for May.

In [ ]:
days = valid["created_at"].dt.date.nunique()
per_day = len(valid) / days
print("May:", days, "days,", round(per_day, 1), "new tickets per day")
print("Capacity, 20% of new tickets:", round(0.2 * per_day, 1), "tickets per day")
print("Escalations per day in May:", round(valid["escalated_72h"].sum() / days, 1))

> **Check.** You should see `38.0` new tickets per day, a capacity of `7.6` tickets per day, and `5.9` escalations per day.

The senior team can review about 7 or 8 new tickets a day. About 6 tickets a day escalate. So a perfect flag list would fit in the capacity. A real model is not perfect: some of its flags are wrong, and those use capacity too.

## 3. One ticket at prediction time

Ticket T-116944 is the first ticket of May that escalated. Run the next cell. It shows the ticket's row, as the model sees it when the ticket is created.

In [ ]:
ticket = valid[valid["ticket_id"] == "T-116944"]
ticket.T

> **Check.** You should see `web_form`, `warranty`, priority `2`, an order of `549.72`, `86` words, and `escalated_72h` = `1`.

Every column except the last one is known at 07:47 on 1 May, when the customer sent the form. The last column, the target, is known only 72 hours later. The model must give its answer at 07:47, from the other columns.

Look at the priority: 2, not 1. A rule "send priority 1 tickets to the senior team" would not flag this ticket.

The file `extra_features.csv` has four more columns for each ticket. Run the next cell to see them for T-116944.

In [ ]:
extra = pd.read_csv("data/extra_features.csv")
extra[extra["ticket_id"] == "T-116944"].T

> **Check.** You should see `prior_escalations_90d` = `0`, `order_value_band` = `high`, `customer_escalation_rate` = `0.1739` and `priority_now` = `1`.

What each column means (the [dataset card](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/data/dataset.md) has the full text):

- `prior_escalations_90d`: the customer's escalations in the 90 days **before** this ticket.
- `order_value_band`: `none`, `low`, `mid` or `high`, from `order_value`.
- `customer_escalation_rate`: the customer's escalation rate over **all** their tickets, including tickets that came later.
- `priority_now`: the priority at the time of the export. The help desk sets it to 1 when a ticket is escalated.

> **Your turn.** Which of the four columns are known when the ticket is created? Ask for each one: when is this value written? Put their names in the list `known_at_creation`. Then run the check cell.

In [ ]:
known_at_creation = ["prior_escalations_90d", "order_value_band"]

In [ ]:
expect_hash('your list of columns known at creation', sorted(known_at_creation) if known_at_creation else None, '519a2e064714cd0c')

`prior_escalations_90d` counts only earlier escalations, and `order_value_band` comes from `order_value`, which exists at creation. Both are fair features. `customer_escalation_rate` uses the targets of later tickets, and `priority_now` is written by the escalation itself. Both are written after the prediction time.

## 4. Failure case: a column from the future

Tomás, the AI engineer, finds `priority_now` and is excited. Run the next cell. It counts the train tickets and escalations for each value of `priority_now`.

> **Predict.** What share of the escalations have `priority_now` = 1? Then run the cell.

In [ ]:
train_extra = train.merge(extra, on="ticket_id", validate="one_to_one")
print(train_extra.groupby("priority_now")["escalated_72h"].agg(["size", "sum", "mean"]).round(3))
caught = train_extra.loc[train_extra["priority_now"] == 1, "escalated_72h"].sum()
print("Escalations with priority_now = 1:", caught, "of", train_extra["escalated_72h"].sum())

> **Check.** You should see `1983` escalations of `1983` with `priority_now` = 1, and no escalation at all with 2 or 3.

A column that finds every escalation looks perfect. That is the warning sign. `priority_now` does not predict the escalation; it records it. When a ticket is created, nobody has escalated it yet, so `priority_now` is the same as `priority`. Run the next cell to see what the "perfect" column really knows at creation.

In [ ]:
escalated = valid[valid["escalated_72h"] == 1]
print("May escalations:", len(escalated))
print("with priority 1 at creation:", (escalated["priority"] == 1).sum())

> **Check.** You should see `183` May escalations, and `61` with priority 1 at creation.

At prediction time, the "perfect" column finds 61 of 183 escalations, one in three. A feature that is written after the prediction time is **leakage**. You met it in the SQL course. In [Select useful features](https://learning.nextia-ai.com/courses/ml/m06/select-useful-features/), you measure what it does to a model's score.

## 5. Your turn: a different decision

Change one thing: the decision. Omar, the product manager, asks: "How many tickets will arrive tomorrow? I plan the agents' shifts the evening before." The unit is now a **day**, not a ticket. The file `daily_tickets.csv` has one row per day.

> **Your turn.** Load `data/daily_tickets.csv` into `daily`. Put the mean of its `tickets` column, rounded to 2 decimals, in `mean_tickets`. Run the cell, then the check cell.

In [ ]:
daily = pd.read_csv("data/daily_tickets.csv")
mean_tickets = round(daily["tickets"].mean(), 2)
print(mean_tickets)

In [ ]:
expect_hash('mean_tickets', mean_tickets, '3610d85cee0bd23d')

Now answer the five questions for Omar's decision. Double-click the next cell to edit it. Then compare with the answer in [Your turn: a different decision](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/#your-turn-a-different-decision) on the lesson page.

| Question | Omar's decision |
|---|---|
| Decision | |
| Prediction unit | |
| Target | |
| Prediction time | |
| Features | |

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson, you turned a need into five answers: the decision, the prediction unit, the target, the prediction time and the features. You saw the data in those terms: 16,939 train tickets with an escalation rate of 11.7%, and 15.5% in May. The senior team can review about 7.6 new tickets a day. A column that is written after the prediction time, such as `priority_now`, can look perfect and find only one in three escalations in real use.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.